# SLM Router v4 — Qwen3 Multi-task Fine-tuning (Colab A100)

v4: Relabeled data + LoRA anti-overfit + Early stopping + Ablation-ready

**Changes vs v3**: LoRA r=32 (was 64), dropout=0.1 (was 0.05), early stopping patience=5, eval every 50 steps  
**Model**: `unsloth/Qwen3-4B` (default) or `unsloth/Qwen3-1.7B` (ablation)  
**Task**: Intent routing (15 classes) + contextual query rewriting  
**Dataset**: multitask_train_v3.jsonl (relabeled, balanced samples)  
**Export**: GGUF Q4_K_M + Q8_0 for laptop deployment via Ollama

In [1]:
%%capture
!pip install "unsloth[colab-new]" "trl>=0.17,<0.19"

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import torch, subprocess
gpu = subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"]).decode().strip()
print(f"GPU: {gpu}")
print(f"bf16 : {torch.cuda.is_bf16_supported()}")
print(f"Torch: {torch.__version__}")
assert torch.cuda.is_bf16_supported(), "A100 required for bf16 training"

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB, 40960 MiB
bf16 : True
Torch: 2.10.0+cu128


In [3]:
# === MODEL SELECTION ===
# Uncomment ONE of these lines:
MODEL_NAME = "unsloth/Qwen3-4B"      # Default: 4B model
# MODEL_NAME = "unsloth/Qwen3-1.7B"  # Ablation: 1.7B model

print(f"Selected model: {MODEL_NAME}")

Selected model: unsloth/Qwen3-4B


In [4]:
import json, re, os, unicodedata
from pathlib import Path
from collections import Counter

from unsloth import FastLanguageModel
import torch
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig

import trl; print(f"TRL version: {trl.__version__}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
TRL version: 0.18.2


In [5]:
# ══════════════════ CONFIG v4 ══════════════════
BASE_MODEL     = MODEL_NAME              # from model selection cell
MAX_SEQ_LENGTH = 1024
LORA_R         = 32      # v4: reduced from 64 -> anti-overfit
LORA_ALPHA     = 64      # v4: keep 2:1 ratio (was 128)
LORA_DROPOUT   = 0.1     # v4: more regularization (was 0.05)
EPOCHS         = 10      # v4: 10 epochs BUT with early stopping
BATCH_SIZE     = 4       # per device (4B needs more VRAM)
GRAD_ACCUM     = 8       # effective batch = 32
LR             = 2e-4    # cosine schedule
WARMUP_RATIO   = 0.06    # slightly more warmup
EVAL_STEPS     = 50      # v4: eval every 50 steps (was per-epoch)

# Paths (Google Drive)
DRIVE_DIR  = Path("/content/drive/MyDrive/hanoi-router-v4")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = str(DRIVE_DIR / "outputs")

# Data — upload multitask_train_v3.jsonl & multitask_val_v3.jsonl to this folder
DATA_DIR   = Path("/content/drive/MyDrive/chatbot-hanoi-weather")
TRAIN_FILE = DATA_DIR / "multitask_train_v3.jsonl"
VAL_FILE   = DATA_DIR / "multitask_val_v3.jsonl"

print(f"Model  : {BASE_MODEL}")
print(f"LoRA   : r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Batch  : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} effective")
print(f"Epochs : {EPOCHS} (with early stopping patience=5)")
print(f"LR     : {LR}")
print(f"Eval   : every {EVAL_STEPS} steps")
print(f"Train  : {TRAIN_FILE}")
print(f"Output : {OUTPUT_DIR}")

Model  : unsloth/Qwen3-4B
LoRA   : r=32, alpha=64, dropout=0.1
Batch  : 4 x 8 = 32 effective
Epochs : 10 (with early stopping patience=5)
LR     : 0.0002
Eval   : every 50 steps
Train  : /content/drive/MyDrive/chatbot-hanoi-weather/multitask_train_v3.jsonl
Output : /content/drive/MyDrive/hanoi-router-v4/outputs


In [6]:
SYSTEM_PROMPT = (
    "Phan loai intent va scope cho cau hoi thoi tiet Ha Noi. Tra ve JSON." + chr(10)
    + chr(10)
    + "## Intents:" + chr(10)
    + "- current_weather: thoi tiet NGAY LUC NAY (nhiet do, troi nang/mua, chung chung)" + chr(10)
    + "- hourly_forecast: dien bien CHI TIET THEO GIO trong ngay (khong chi hoi mua)" + chr(10)
    + "- daily_forecast: du bao NHIEU NGAY toi (3 ngay, tuan toi, cuoi tuan)" + chr(10)
    + "- weather_overview: TONG QUAN, tom tat thoi tiet hom nay/ngay mai (khong hoi thong so cu the)" + chr(10)
    + "- rain_query: hoi CO MUA KHONG, xac suat mua, mua bao lau/luc nao tanh" + chr(10)
    + "- temperature_query: hoi CU THE VE NHIET DO (bao nhieu do, nong/lanh)" + chr(10)
    + "- wind_query: hoi CU THE VE GIO (gio manh khong, huong gio, toc do gio)" + chr(10)
    + "- humidity_fog_query: hoi ve DO AM, SUONG MU, suong muoi" + chr(10)
    + "- historical_weather: thoi tiet NGAY/TUAN TRUOC, du lieu QUA KHU" + chr(10)
    + "- location_comparison: SO SANH thoi tiet giua cac quan/phuong/dia diem" + chr(10)
    + "- activity_weather: thoi tiet PHU HOP DE LAM hoat dong X khong (chay bo, picnic)" + chr(10)
    + "- expert_weather_param: thong so KY THUAT chuyen sau (ap suat, UV, diem suong, tam nhin)" + chr(10)
    + "- weather_alert: CANH BAO nguy hiem: bao/ap thap, ngap, giong/loc, ret hai, nang nong cuc doan" + chr(10)
    + "- seasonal_context: SO SANH voi hom qua/tuan truoc, xu huong, bat thuong theo MUA" + chr(10)
    + "- smalltalk_weather: chao hoi, ngoai pham vi, cau hoi khong lien quan thoi tiet" + chr(10)
    + chr(10)
    + "## Anti-confusion rules:" + chr(10)
    + "- bay gio/bay gio = thoi diem hien tai -> current_weather (KHONG phai wind_query)" + chr(10)
    + "- gio/gio manh/toc do gio = yeu to gio -> wind_query" + chr(10)
    + "- bao/lu/canh bao -> weather_alert (KHONG phai rain_query)" + chr(10)
    + chr(10)
    + "## Scopes:" + chr(10)
    + "- city: toan Ha Noi hoac khong noi ro dia diem" + chr(10)
    + "- district: quan/huyen hoac dia diem noi tieng (Ho Guom->Hoan Kiem)" + chr(10)
    + "- ward: phuong/xa" + chr(10)
    + chr(10)
    + "## Output:" + chr(10)
    + '{"intent":"...","scope":"...","confidence":0.9}' + chr(10)
    + "Them rewritten_query neu co context va cau thieu dia diem."
)

print(SYSTEM_PROMPT[:200], "...")
print(f"Prompt length: {len(SYSTEM_PROMPT)} chars")

Phan loai intent va scope cho cau hoi thoi tiet Ha Noi. Tra ve JSON.

## Intents:
- current_weather: thoi tiet NGAY LUC NAY (nhiet do, troi nang/mua, chung chung)
- hourly_forecast: dien bien CHI TIET ...
Prompt length: 1721 chars


In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,
    load_in_4bit   = False,  # A100 has enough VRAM for full bf16
)
print(f"Model loaded: {BASE_MODEL}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-4B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: unsloth/Qwen3-4B
Parameters: 4,022,468,096


In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,         # v4: 32 (was 64) -> reduce overfit
    lora_alpha     = LORA_ALPHA,     # v4: 64 (was 128) -> keep 2:1 ratio
    lora_dropout   = LORA_DROPOUT,   # v4: 0.1 (was 0.05) -> more regularization
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.17 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable: 66,060,288 / 4,088,528,384 (1.62%)


In [9]:
# Manual tokenizer setup (skip get_chat_template — Unsloth Qwen3 bugs)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

# Verify ChatML tokens
test_ids = tokenizer.encode("<|im_start|>", add_special_tokens=False)
print(f"pad_token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"<|im_start|> ids: {test_ids}")
assert len(test_ids) == 1, "ChatML special tokens not found"

pad_token: '<|PAD_TOKEN|>' (id=151669)
<|im_start|> ids: [151644]


In [10]:
VALID_INTENTS = [
    "current_weather", "hourly_forecast", "daily_forecast", "weather_overview",
    "rain_query", "temperature_query", "wind_query", "humidity_fog_query",
    "historical_weather", "location_comparison", "activity_weather",
    "expert_weather_param", "weather_alert", "seasonal_context", "smalltalk_weather",
]
VALID_SCOPES = ["city", "district", "ward"]

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_raw = load_jsonl(TRAIN_FILE)
val_raw   = load_jsonl(VAL_FILE)

def validate(records, name):
    valid = []
    for r in records:
        out = r["output"] if isinstance(r["output"], dict) else json.loads(r["output"])
        if out["intent"] in VALID_INTENTS and out["scope"] in VALID_SCOPES:
            valid.append(r)
    print(f"{name}: {len(valid)}/{len(records)} valid")
    return valid

train_raw = validate(train_raw, "Train")
val_raw   = validate(val_raw, "Val")

intents = Counter(r["output"]["intent"] if isinstance(r["output"], dict) else json.loads(r["output"])["intent"] for r in train_raw)
ctx_count = sum(1 for r in train_raw if r.get("context"))
rw_count  = sum(1 for r in train_raw if (r["output"] if isinstance(r["output"], dict) else json.loads(r["output"])).get("rewritten_query"))
print(f"Context records: {ctx_count} ({ctx_count/len(train_raw)*100:.1f}%)")
print(f"Rewrite records: {rw_count} ({rw_count/len(train_raw)*100:.1f}%)")
for intent, count in sorted(intents.items(), key=lambda x: -x[1]):
    print(f"  {intent:<25} {count:>5} ({count/len(train_raw)*100:.1f}%)")

Train: 4005/4005 valid
Val: 672/672 valid
Context records: 810 (20.2%)
Rewrite records: 642 (16.0%)
  activity_weather            316 (7.9%)
  rain_query                  306 (7.6%)
  weather_alert               303 (7.6%)
  temperature_query           293 (7.3%)
  current_weather             287 (7.2%)
  wind_query                  281 (7.0%)
  daily_forecast              267 (6.7%)
  hourly_forecast             267 (6.7%)
  weather_overview            257 (6.4%)
  location_comparison         256 (6.4%)
  seasonal_context            255 (6.4%)
  historical_weather          255 (6.4%)
  humidity_fog_query          255 (6.4%)
  expert_weather_param        254 (6.3%)
  smalltalk_weather           153 (3.8%)


In [11]:
IM_START = "<|im_start|>"
IM_END   = "<|im_end|>"
NL       = chr(10)

def format_record(rec):
    user_msg = str(rec.get("input", "")).strip()
    ctx = rec.get("context")
    if ctx:
        ctx_str  = json.dumps(ctx, ensure_ascii=False, separators=(",", ":"))
        user_msg = "[CONTEXT: " + ctx_str + "]" + NL + user_msg
    out = rec.get("output", {})
    if isinstance(out, str): out = json.loads(out)
    output_dict = {
        "intent":     out["intent"],
        "scope":      out["scope"],
        "confidence": round(float(out.get("confidence", 0.9)), 2),
    }
    rw = out.get("rewritten_query")
    if rw and str(rw).strip():
        output_dict["rewritten_query"] = str(rw).strip()
    text  = IM_START + "system"    + NL + SYSTEM_PROMPT + IM_END + NL
    text += IM_START + "user"      + NL + user_msg      + IM_END + NL
    text += IM_START + "assistant" + NL + json.dumps(output_dict, ensure_ascii=False) + IM_END + NL
    return text

sample = format_record(train_raw[0])
print(sample[:400])
print("...")
print(f"Tokens: {len(tokenizer.encode(sample))}")

train_texts = [format_record(r) for r in train_raw]
val_texts   = [format_record(r) for r in val_raw]

lengths = [len(tokenizer.encode(t)) for t in train_texts[:200]]
print(f"Avg tokens: {sum(lengths)/len(lengths):.0f}, Max: {max(lengths)}, Over {MAX_SEQ_LENGTH}: {sum(1 for l in lengths if l > MAX_SEQ_LENGTH)}")

raw_train = Dataset.from_dict({"text": train_texts})
raw_val   = Dataset.from_dict({"text": val_texts})
print(f"Train: {len(raw_train)}, Val: {len(raw_val)}")

<|im_start|>system
Phan loai intent va scope cho cau hoi thoi tiet Ha Noi. Tra ve JSON.

## Intents:
- current_weather: thoi tiet NGAY LUC NAY (nhiet do, troi nang/mua, chung chung)
- hourly_forecast: dien bien CHI TIET THEO GIO trong ngay (khong chi hoi mua)
- daily_forecast: du bao NHIEU NGAY toi (3 ngay, tuan toi, cuoi tuan)
- weather_overview: TONG QUAN, tom tat thoi tiet hom nay/ngay mai (kho
...
Tokens: 667
Avg tokens: 667, Max: 702, Over 1024: 0
Train: 4005, Val: 672


In [12]:
# v3 KEY FIX (preserved in v4): Completion-only loss masking
# Only compute loss on assistant response tokens, NOT system prompt / user query.
# This reduces loss from ~13.6 (full-seq) to ~0.1 (response-only) and
# dramatically improves accuracy because the model focuses on learning
# the classification output, not memorizing the system prompt.

ASSISTANT_MARKER = "<|im_start|>assistant" + chr(10)
assistant_ids = tokenizer.encode(ASSISTANT_MARKER, add_special_tokens=False)
print(f"Assistant marker token IDs: {assistant_ids}")
print(f"Assistant marker length: {len(assistant_ids)} tokens")

def tokenize_with_masking(examples):
    """Tokenize and mask labels: -100 for everything before assistant response."""
    enc = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
        return_tensors=None,
    )
    all_labels = []
    masked_count = 0
    for input_ids in enc["input_ids"]:
        labels = [-100] * len(input_ids)  # mask everything by default
        # Find the assistant marker position
        marker_len = len(assistant_ids)
        found = False
        for i in range(len(input_ids) - marker_len + 1):
            if input_ids[i:i+marker_len] == assistant_ids:
                # Unmask everything AFTER the marker (the actual response)
                start = i + marker_len
                for j in range(start, len(input_ids)):
                    labels[j] = input_ids[j]
                found = True
                masked_count += 1
                break
        if not found:
            # Fallback: unmask everything (should not happen)
            labels = list(input_ids)
        all_labels.append(labels)
    enc["labels"] = all_labels
    return enc

train_dataset = raw_train.map(tokenize_with_masking, batched=True, remove_columns=["text"])
val_dataset   = raw_val.map(tokenize_with_masking, batched=True, remove_columns=["text"])

# Verify masking works correctly
sample_labels = train_dataset[0]["labels"]
sample_ids    = train_dataset[0]["input_ids"]
n_masked = sum(1 for l in sample_labels if l == -100)
n_total  = len(sample_labels)
n_active = n_total - n_masked
print(f"Train tokenized: {len(train_dataset)} samples")
print(f"Val tokenized  : {len(val_dataset)} samples")
print(f"Sample: {n_total} tokens total, {n_masked} masked (-100), {n_active} active")
print(f"Masking ratio: {n_masked/n_total*100:.1f}% masked (system+user), {n_active/n_total*100:.1f}% active (assistant)")

# Decode the active part to verify
active_ids = [sample_ids[i] for i in range(len(sample_labels)) if sample_labels[i] != -100]
print(f"Active tokens decoded: {tokenizer.decode(active_ids)[:200]}")

Assistant marker token IDs: [151644, 77091, 198]
Assistant marker length: 3 tokens


Map:   0%|          | 0/4005 [00:00<?, ? examples/s]

Map:   0%|          | 0/672 [00:00<?, ? examples/s]

Train tokenized: 4005 samples
Val tokenized  : 672 samples
Sample: 667 tokens total, 643 masked (-100), 24 active
Masking ratio: 96.4% masked (system+user), 3.6% active (assistant)
Active tokens decoded: {"intent": "activity_weather", "scope": "ward", "confidence": 0.89}<|im_end|>



In [13]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer          = tokenizer,
    model              = model,
    label_pad_token_id = -100,
    pad_to_multiple_of = 8,
)
print("DataCollatorForSeq2Seq ready")

DataCollatorForSeq2Seq ready


In [14]:
training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    warmup_ratio                = WARMUP_RATIO,
    lr_scheduler_type           = "cosine",          # cosine annealing
    logging_steps               = 10,
    eval_strategy               = "steps",           # v4: was "epoch"
    eval_steps                  = EVAL_STEPS,        # v4: every 50 steps
    save_strategy               = "steps",           # v4: was "epoch"
    save_steps                  = EVAL_STEPS,        # v4: save aligned with eval
    save_total_limit            = 3,
    metric_for_best_model       = "eval_loss",
    load_best_model_at_end      = True,
    bf16                        = True,
    fp16                        = False,
    optim                       = "adamw_torch_fused",
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,
    max_seq_length              = MAX_SEQ_LENGTH,
    # v4: NO label_smoothing_factor (confirmed removed in v3)
)
print("SFTConfig ready")
steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
print(f"Steps/epoch: {steps_per_epoch}  |  Total: {total_steps}")
print(f"Warmup steps: {int(total_steps * WARMUP_RATIO)}")
print(f"Eval every {EVAL_STEPS} steps  |  ~{steps_per_epoch // EVAL_STEPS} evals per epoch")
print(f"LR scheduler: cosine")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


SFTConfig ready
Steps/epoch: 125  |  Total: 1250
Warmup steps: 75
Eval every 50 steps  |  ~2 evals per epoch
LR scheduler: cosine


In [15]:
# v4: Early stopping to prevent overfitting
early_stop = EarlyStoppingCallback(
    early_stopping_patience   = 5,      # stop after 5 evals with no improvement
    early_stopping_threshold  = 0.001,   # minimum improvement to count
)
print(f"EarlyStoppingCallback: patience=5, threshold=0.001")
print(f"Will stop if eval_loss does not improve by 0.001 for 5 consecutive evals ({5 * EVAL_STEPS} steps)")

EarlyStoppingCallback: patience=5, threshold=0.001
Will stop if eval_loss does not improve by 0.001 for 5 consecutive evals (250 steps)


In [16]:
trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    data_collator    = data_collator,
    packing          = False,
    callbacks        = [early_stop],  # v4: early stopping
)
print("SFTTrainer ready (with EarlyStoppingCallback)")

SFTTrainer ready (with EarlyStoppingCallback)


In [17]:
trainer_stats = trainer.train()
print(f"Train loss: {trainer_stats.training_loss:.4f}")
print(f"Runtime:    {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Samples/s:  {trainer_stats.metrics['train_samples_per_second']:.1f}")
print(f"Stopped at step: {trainer.state.global_step} / {total_steps}")
if trainer.state.global_step < total_steps:
    print("Early stopping triggered!")
else:
    print("Ran all epochs (no early stop)")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,005 | Num Epochs = 10 | Total steps = 1,260
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Step,Training Loss,Validation Loss
50,0.168390,0.145102
100,0.117384,0.119656
150,0.106695,0.116975
200,0.113518,0.110257
250,0.093437,0.103048
300,0.084323,0.096204
350,0.082563,0.095621
400,0.069070,0.097716
450,0.067778,0.094058
500,0.067741,0.093917


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Step,Training Loss,Validation Loss
50,0.168390,0.145102
100,0.117384,0.119656
150,0.106695,0.116975
200,0.113518,0.110257
250,0.093437,0.103048
300,0.084323,0.096204
350,0.082563,0.095621
400,0.069070,0.097716
450,0.067778,0.094058
500,0.067741,0.093917


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Train loss: 0.0938
Runtime:    4231s
Samples/s:  9.5
Stopped at step: 1000 / 1250
Early stopping triggered!


In [18]:
adapter_dir = DRIVE_DIR / "lora_adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter saved to {adapter_dir}")

Adapter saved to /content/drive/MyDrive/hanoi-router-v4/lora_adapter


In [19]:
# v4: Eval with per-intent confusion matrix
model.eval()

TEST_FILE = DATA_DIR / "multitask_val_v3.jsonl"
if not TEST_FILE.exists():
    TEST_FILE = VAL_FILE
test_records = []
with open(TEST_FILE, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_records.append(json.loads(line))
print(f"Test records: {len(test_records)}")

NL_eval = chr(10)

correct_route  = 0
total_route    = 0
rw_correct     = 0
rw_total       = 0
rw_entity_ok   = 0
norw_correct   = 0
norw_total     = 0

# Per-intent tracking
intent_correct = Counter()
intent_total   = Counter()
confusion_pairs = Counter()   # (expected, predicted)

for rec in test_records:
    out = rec["output"] if isinstance(rec["output"], dict) else json.loads(rec["output"])
    expected_intent = out["intent"]
    expected_scope  = out["scope"]
    has_rw = bool(out.get("rewritten_query"))

    user_msg = str(rec.get("input", "")).strip()
    ctx = rec.get("context")
    if ctx:
        ctx_str  = json.dumps(ctx, ensure_ascii=False, separators=(",",":"))
        user_msg = "[CONTEXT: " + ctx_str + "]" + NL_eval + user_msg

    prompt  = "<|im_start|>system" + NL_eval + SYSTEM_PROMPT + "<|im_end|>" + NL_eval
    prompt += "<|im_start|>user"   + NL_eval + user_msg      + "<|im_end|>" + NL_eval
    prompt += "<|im_start|>assistant" + NL_eval

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=128, temperature=0.0,
            do_sample=False, use_cache=False,
        )
    raw = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if "</think>" in raw:
        raw = raw[raw.rfind("</think>") + len("</think>"):].strip()

    try:
        pred = json.loads(raw)
    except json.JSONDecodeError:
        intent_total[expected_intent] += 1
        total_route += 1
        continue

    pred_intent = pred.get("intent", "")
    pred_scope  = pred.get("scope", "")
    route_ok = (pred_intent == expected_intent and pred_scope == expected_scope)
    intent_total[expected_intent] += 1
    if route_ok:
        correct_route += 1
        intent_correct[expected_intent] += 1
    else:
        confusion_pairs[(expected_intent, pred_intent)] += 1
    total_route += 1

    if has_rw:
        rw_total += 1
        if route_ok:
            rw_correct += 1
        pred_rw = pred.get("rewritten_query", "")
        if ctx and ctx.get("location") and ctx["location"].lower() in pred_rw.lower():
            rw_entity_ok += 1
    else:
        norw_total += 1
        if route_ok:
            norw_correct += 1

print("=" * 60)
print("FULL EVAL RESULTS (v4)")
print("=" * 60)
ra = correct_route / total_route if total_route else 0
print(f"Routing accuracy    : {correct_route}/{total_route} = {ra:.1%}    target >= 92%")
if rw_total:
    print(f"Rewrite routing acc : {rw_correct}/{rw_total} = {rw_correct/rw_total:.1%}    target >= 85%")
    print(f"Entity coverage     : {rw_entity_ok}/{rw_total} = {rw_entity_ok/rw_total:.1%}    target >= 70%")
if norw_total:
    print(f"No-rewrite routing  : {norw_correct}/{norw_total} = {norw_correct/norw_total:.1%}    target >= 80%")

verdict = "PASS" if ra >= 0.92 else "FAIL"
print("=" * 60)
print("Pass routing>=92% : " + verdict)
print("=" * 60)

# Per-intent accuracy
print()
print("PER-INTENT ACCURACY:")
print(f"  {'Intent':<25} {'Correct':>7} {'Total':>7} {'Acc':>7}")
print(f"  {'-'*50}")
for intent in sorted(VALID_INTENTS):
    t = intent_total.get(intent, 0)
    c = intent_correct.get(intent, 0)
    acc = c / t if t else 0
    flag = " <<<" if acc < 0.85 else ""
    print(f"  {intent:<25} {c:>7} {t:>7} {acc:>6.1%}{flag}")

# Top confusion pairs
if confusion_pairs:
    print()
    print("TOP CONFUSION PAIRS (expected -> predicted):")
    for (exp, pred), count in confusion_pairs.most_common(10):
        print(f"  {exp:<25} -> {pred:<25} x{count}")

Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test records: 672


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

FULL EVAL RESULTS (v4)
Routing accuracy    : 638/672 = 94.9%    target >= 92%
Rewrite routing acc : 118/122 = 96.7%    target >= 85%
Entity coverage     : 119/122 = 97.5%    target >= 70%
No-rewrite routing  : 520/550 = 94.5%    target >= 80%
Pass routing>=92% : PASS

PER-INTENT ACCURACY:
  Intent                    Correct   Total     Acc
  --------------------------------------------------
  activity_weather               56      56 100.0%
  current_weather                37      41  90.2%
  daily_forecast                 42      45  93.3%
  expert_weather_param           42      43  97.7%
  historical_weather             45      45 100.0%
  hourly_forecast                40      45  88.9%
  humidity_fog_query             45      45 100.0%
  location_comparison            42      44  95.5%
  rain_query                     43      45  95.6%
  seasonal_context               44      45  97.8%
  smalltalk_weather              20      25  80.0% <<<
  temperature_query              47     

In [23]:
NL_inf = chr(10)
test_cases = [
    {"name": "Basic current",   "input": "Bay gio thoi tiet Cau Giay the nao?",
     "expected_intent": "current_weather", "context": None},
    {"name": "Daily forecast",  "input": "Cuoi tuan Ha Noi the nao?",
     "expected_intent": "daily_forecast",  "context": None},
    {"name": "Wind query",      "input": "Gio o Hoang Mai manh khong?",
     "expected_intent": "wind_query",      "context": None},
    {"name": "Weather alert",   "input": "Co bao o Ha Noi khong?",
     "expected_intent": "weather_alert",   "context": None},
    {"name": "Context rewrite", "input": "Con ngay mai?",
     "expected_intent": "daily_forecast",
     "context": {"location": "Cau Giay", "intent": "current_weather", "turn": 1}},
    {"name": "Activity",        "input": "Sang mai di chay bo duoc khong?",
     "expected_intent": "activity_weather", "context": None},
]

print("=" * 60)
print("INFERENCE TEST (v4)")
print("=" * 60)
all_pass = True
for tc in test_cases:
    user_msg = tc["input"]
    ctx = tc.get("context")
    if ctx:
        ctx_str  = json.dumps(ctx, ensure_ascii=False, separators=(",",":"))
        user_msg = "[CONTEXT: " + ctx_str + "]" + NL_inf + user_msg
    prompt  = "<|im_start|>system"    + NL_inf + SYSTEM_PROMPT + "<|im_end|>" + NL_inf
    prompt += "<|im_start|>user"      + NL_inf + user_msg      + "<|im_end|>" + NL_inf
    prompt += "<|im_start|>assistant" + NL_inf
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=128, temperature=0.0, do_sample=False, use_cache=False)
    raw = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if "</think>" in raw:
        raw = raw[raw.rfind("</think>") + len("</think>"):].strip()
    try:
        pred = json.loads(raw)
    except Exception:
        pred = {"raw": raw}
    ok = pred.get("intent") == tc["expected_intent"]
    status = "OK" if ok else "FAIL"
    if not ok:
        all_pass = False
    name = tc["name"]
    inp_text = tc["input"][:60]
    print(status.rjust(4) + " [" + name + "]")
    print("     Input:  " + inp_text)
    print("     Pred:   " + json.dumps(pred, ensure_ascii=False))
    print()
print("All passed: " + str(all_pass))

Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFERENCE TEST (v4)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  OK [Basic current]
     Input:  Bay gio thoi tiet Cau Giay the nao?
     Pred:   {"intent": "current_weather", "scope": "district", "confidence": 0.9}



Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  OK [Wind query]
     Input:  Gio o Hoang Mai manh khong?
     Pred:   {"intent": "wind_query", "scope": "district", "confidence": 0.91}



Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  OK [Weather alert]
     Input:  Co bao o Ha Noi khong?
     Pred:   {"intent": "weather_alert", "scope": "city", "confidence": 0.9}



Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  OK [Context rewrite]
     Input:  Con ngay mai?
     Pred:   {"intent": "daily_forecast", "scope": "district", "confidence": 0.89, "rewritten_query": "Ngày mai ở huyện Cầu Giấy thời tiết thế nào?"}

  OK [Activity]
     Input:  Sang mai di chay bo duoc khong?
     Pred:   {"intent": "activity_weather", "scope": "city", "confidence": 0.9}

All passed: True


In [24]:
# Free disk space before export
import shutil, gc
output_path = Path(OUTPUT_DIR)
if output_path.exists():
    for ckpt in sorted(output_path.glob("checkpoint-*")):
        print(f"Deleting {ckpt.name}...")
        shutil.rmtree(ckpt)
torch.cuda.empty_cache()
gc.collect()

# Export Q4_K_M to Drive
gguf_dir_q4 = str(DRIVE_DIR / "gguf" / "q4_k_m")
print("Exporting GGUF Q4_K_M...")
model.save_pretrained_gguf(
    gguf_dir_q4,
    tokenizer,
    quantization_method="q4_k_m",
)
print("Q4_K_M done!")

for f in Path(gguf_dir_q4).iterdir():
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.1f} MB")

# Export Q8_0 to Drive
gguf_dir_q8 = str(DRIVE_DIR / "gguf" / "q8_0")
print()
print("Exporting GGUF Q8_0...")
model.save_pretrained_gguf(
    gguf_dir_q8,
    tokenizer,
    quantization_method="q8_0",
)
print("Q8_0 done!")

for f in Path(gguf_dir_q8).iterdir():
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.1f} MB")

Exporting GGUF Q4_K_M...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 2 files from cache to `/content/drive/MyDrive/hanoi-router-v4/gguf/q4_k_m`:   0%|          | 0/2 [00:00<?, ?it/s]
Unsloth: Copying 2 files from cache to `/content/drive/MyDrive/hanoi-router-v4/gguf/q4_k_m`:  50%|█████     | 1/2 [00:18<00:18, 18.70s/it]
Unsloth: Copying 2 files from cache to `/content/drive/MyDrive/hanoi-router-v4/gguf/q4_k_m`: 100%|██████████| 2/2 [00:39<00:00, 19.75s/it]


Successfully copied all 2 files from cache to `/content/drive/MyDrive/hanoi-router-v4/gguf/q4_k_m`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 15477.14it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:50<00:00, 25.41s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/hanoi-router-v4/gguf/q4_k_m`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/hanoi-router-v4/gguf/q4_k_m_gguf/Qwen3-4B.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/hanoi-router-v4/gguf/q4_k_m_gguf/Qwen3-4B.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/hanoi-router-v4/gguf/q4_k_m_gguf/Qwen3-4B.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/drive/MyDrive/hanoi-router-v4/gguf/q4_k_m_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/drive/MyDrive/hanoi-router-v4/gguf/q4_k_m_gguf/Modelfile
Q4_K_M done!
  chat_template.jinja: 0.0 MB
  tokenizer_config.json: 0.0 MB
  tokenizer.json: 10

RuntimeError: Failed to save/merge model: Unsloth: Failed saving locally - no disk space left. Uploading can work luckily! Use .push_to_hub instead.

In [ ]:
modelfile_content = (
    "FROM ./unsloth.Q4_K_M.gguf" + chr(10)
    + chr(10)
    + "TEMPLATE " + chr(34)*3 + chr(10)
    + "{{- if .System }}<|im_start|>system" + chr(10)
    + "{{ .System }}<|im_end|>" + chr(10)
    + "{{ end }}<|im_start|>user" + chr(10)
    + "{{ .Prompt }}<|im_end|>" + chr(10)
    + "<|im_start|>assistant" + chr(10)
    + "{{ .Response }}<|im_end|>" + chr(10)
    + chr(34)*3 + chr(10)
    + chr(10)
    + "PARAMETER temperature 0" + chr(10)
    + "PARAMETER num_predict 128" + chr(10)
    + "PARAMETER stop " + chr(34) + "<|im_end|>" + chr(34) + chr(10)
    + "PARAMETER stop " + chr(34) + "<|im_start|>" + chr(34) + chr(10)
)

modelfile_path = DRIVE_DIR / "gguf" / "q4_k_m" / "Modelfile"
modelfile_path.write_text(modelfile_content, encoding="utf-8")
print(f"Modelfile saved to {modelfile_path}")
print(modelfile_content)

In [ ]:
# FALLBACK: If disk is full, push directly to HuggingFace
# Uncomment below to use:

from google.colab import userdata
hf_token = ""
HF_REPO = "daredevil467/qwen3-4b-hanoi-weather-router-v4"

model.push_to_hub_gguf(
    HF_REPO,
    tokenizer,
    quantization_method="q4_k_m",
    token=hf_token,
)
print(f"Pushed GGUF Q4_K_M to https://huggingface.co/{HF_REPO}")

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 2 files from cache to `/tmp/unsloth_gguf_tbadycv4`:   0%|          | 0/2 [00:00<?, ?it/s]
Unsloth: Copying 2 files from cache to `/tmp/unsloth_gguf_tbadycv4`:  50%|█████     | 1/2 [00:10<00:10, 10.21s/it]
Unsloth: Copying 2 files from cache to `/tmp/unsloth_gguf_tbadycv4`: 100%|██████████| 2/2 [00:18<00:00,  9.32s/it]


Successfully copied all 2 files from cache to `/tmp/unsloth_gguf_tbadycv4`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_tbadycv4`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_tbadycv4_gguf/Qwen3-4B.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_tbadycv4_gguf/Qwen3-4B.Q4_K_M.gguf']
Unsloth: example usage for tex

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...gguf/Qwen3-4B.Q4_K_M.gguf:   7%|6         |  168MB / 2.50GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/daredevil467/qwen3-4b-hanoi-weather-router-v4
Unsloth: Cleaning up temporary files...
Pushed GGUF Q4_K_M to https://huggingface.co/daredevil467/qwen3-4b-hanoi-weather-router-v4


In [27]:
print("=" * 60)
print("  TRAINING SUMMARY v4")
print("=" * 60)
print(f"  Model      : {BASE_MODEL}")
print(f"  LoRA       : r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"  Dataset    : {len(train_dataset)} train / {len(val_dataset)} val")
print(f"  Epochs     : {EPOCHS} (early stop patience=5)")
print(f"  Stopped at : step {trainer.state.global_step} / {total_steps}")
print(f"  Train loss : {trainer_stats.training_loss:.4f}")
print(f"  Key fixes  : completion-only masking + anti-overfit LoRA + early stopping")
print(f"  Output     : {DRIVE_DIR}")
print("=" * 60)
print()
print("Next steps:")
print("  1. Download GGUF Q4_K_M from Drive")
print("  2. ollama create hanoi-weather-router-v4 -f Modelfile")
print("  3. Test: ollama run hanoi-weather-router-v4")
print("  4. Run ablation with MODEL_NAME=unsloth/Qwen3-1.7B and/or different LoRA ranks")

  TRAINING SUMMARY v4
  Model      : unsloth/Qwen3-4B
  LoRA       : r=32, alpha=64, dropout=0.1
  Dataset    : 4005 train / 672 val
  Epochs     : 10 (early stop patience=5)
  Stopped at : step 1000 / 1250
  Train loss : 0.0938
  Key fixes  : completion-only masking + anti-overfit LoRA + early stopping
  Output     : /content/drive/MyDrive/hanoi-router-v4

Next steps:
  1. Download GGUF Q4_K_M from Drive
  2. ollama create hanoi-weather-router-v4 -f Modelfile
  3. Test: ollama run hanoi-weather-router-v4
  4. Run ablation with MODEL_NAME=unsloth/Qwen3-1.7B and/or different LoRA ranks


## LoRA Rank Ablation (Optional)

To run the ablation study, re-run this notebook with different values:

| LoRA r | ~Trainable Params | Notes |
|--------|-------------------|-------|
| r=8    | ~8.5M params      | Minimal capacity |
| r=16   | ~17M params       | Lightweight |
| r=32   | ~34M params       | **Default (v4)** |
| r=64   | ~67M params       | v3 default (higher overfit risk) |

**Record for each run**: final `train_loss`, best `val_loss`, `routing_accuracy`, per-intent accuracy

### How to run:
1. Change `LORA_R` and `LORA_ALPHA` (keep 2:1 ratio) in the config cell
2. Restart runtime and run all cells
3. Compare eval results across runs

### Model ablation:
- Switch `MODEL_NAME` in the model selection cell to `unsloth/Qwen3-1.7B`
- Compare 4B vs 1.7B at the same LoRA rank